# Step 5 — Two-tower model (query + user tower, item tower)

Trains the advanced retrieval model on GPU with cross-fitting and scores every displayed item.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [ ]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/harvestingmoon/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

## 5.1 Architecture (`src/jds/twotower/model.py`)

* **Item tower:** mean term embeddings of title (20 terms), brand (4) and category L3+L4 names (6); brand, shop and 4
  category id embeddings; popularity buckets of clicks (7 d), carts (90 d), orders (365 d) → MLP → 128-d. No item-id
  embedding (most displayed items are cold).
* **Query + user tower:** the query shares the term embedding. Each history event = item-tower output + event type +
  log time before the query + the query it was issued under. The history is split into **short (< 7 d, all types),
  mid (7–90 d, carts/follows/orders) and long (≥ 90 d, orders only)** segments — the retention windows — each pooled by
  its own query-guided attention with a null slot → MLP → 128-d. Score = cosine similarity.
* **Loss:** listwise softmax over each training slate with target ∝ 2^label − 1, plus an auxiliary in-batch softmax on
  historical (query → item) events from the last 7 days, weighted by event type.

## 5.2 Cross-fitting (`src/jds/twotower/train.py`)

The ranker (step 6) learns on training users, so their tower scores must come from towers that never saw their slates:
`fold0` is trained on fold 1 and scores fold 0, and vice versa; `full` is trained on all training users and scores
validation/test. Best epoch by validation graded NDCG@10. On a T4/V100, float16 AMP is used automatically; if CUDA runs
out of memory, see the README troubleshooting section.

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU will be very slow)")

In [ ]:
run.stage("03_train_tower.py")

In [ ]:
run.stage("04_score_tower.py")

In [ ]:
banner()
import pandas as pd, json
for f in sorted(run.p.tower.glob("*_history.json")):
    print(f.stem, "(validation metrics by epoch)")
    display(pd.DataFrame(json.loads(f.read_text()))[["epoch", "train_loss", "mrr", "ndcg10", "gndcg10"]].style.format(precision=4).highlight_max(subset=["gndcg10"], color="#c6efce"))

In [ ]:
banner()
pd.DataFrame(run.json("tower.json")).style.format(precision=4)

## 5.3 Optional tower ablations

Train the full tower without the history (query-only) or without the query (user-only) to measure what each side adds.
Each takes about as long as one tower model.

In [ ]:
RUN_TOWER_ABLATIONS = False
if RUN_TOWER_ABLATIONS:
    for v in ("no_history", "no_query"):
        run.stage("03_train_tower.py", "--variant", v)
        run.stage("04_score_tower.py", "--variant", v)
for v in ("no_history", "no_query"):
    f = run.results(f"tower_{v}.json")
    if f.exists():
        display(pd.DataFrame(json.loads(f.read_text())))